In [12]:
import pandas as pd
import numpy as np
import json, os

In [14]:
features = pd.read_csv('../data/processed/features_engineered.csv')

In [15]:
features.columns

Index(['area', 'year', 'quarter', 'latitude', 'longitude', 'avg_d_mbps',
       'avg_u_mbps', 'avg_lat_ms', 'tests', 'nearest_tower_distance_km',
       'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
       'visible_tower_count_5km', 'digital_elevation_model', 'region_enc',
       'typeOfArea_enc', 'city_enc', 'demand_growth_pct', 'region',
       'typeOfArea', 'city', 'block_number', 'tower_density_ratio',
       'year_trend', 'area_rolling_d_mbps', 'area_median_d', 'area_median_u',
       'area_median_lat', 'area_test_count', 'distance_x_density'],
      dtype='object')

In [16]:
features.head(2)

,area,year,quarter,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,nearest_tower_distance_km,...,city,block_number,tower_density_ratio,year_trend,area_rolling_d_mbps,area_median_d,area_median_u,area_median_lat,area_test_count,distance_x_density
0,A'ali,2020,1,26.167764,50.516785,34.6080,6.629200,21.200000,52,0.910574,...,A'ali,738,0.017297,0,152.420662,178.857833,23.14375,20.416667,2070,0.004922
1,A'ali,2020,2,26.168586,50.516052,24.7855,8.958833,23.166667,105,0.949577,...,A'ali,714,0.014235,0,34.608000,178.857833,23.14375,20.416667,2070,0.005069


In [17]:
feature_cols = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests',
    'area_median_d',
    'area_median_u', 
    'area_median_lat',
    'area_test_count',
    'distance_x_density',
]

In [260]:
import json
import os

feature_cols = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests',
    'area_median_d',
    'area_median_u', 
    'area_median_lat',
    'area_test_count',
    'distance_x_density',
]

path = os.path.join('..', 'outputs', 'models', 'feature_cols2.json')

with open(path, 'w') as f:
    json.dump(feature_cols, f, indent=2)

In [18]:
targets = ['avg_d_mbps','avg_u_mbps', 'avg_lat_ms']

In [19]:
X = features[feature_cols]
y = features[targets]

In [20]:
X.shape

(6086, 21)

In [21]:
y.shape

(6086, 3)

In [25]:
X.isnull().sum()

latitude                     0
longitude                    0
nearest_tower_distance_km    0
tower_count_1km              0
tower_count_2km              0
tower_count_5km              0
tower_density_ratio          0
digital_elevation_model      0
region_enc                   0
typeOfArea_enc               0
city_enc                     0
demand_growth_pct            0
year_trend                   0
quarter                      0
area_rolling_d_mbps          0
tests                        0
area_median_d                0
area_median_u                0
area_median_lat              0
area_test_count              0
distance_x_density           0
dtype: int64

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, random_state=42, test_size=0.15)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15, random_state=42)

In [31]:
print(f"Train: {X_train.shape[0]}| val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

Train: 4397| val: 776 | Test: 913


In [37]:
X.dtypes

latitude                     float64
longitude                    float64
nearest_tower_distance_km    float64
tower_count_1km              float64
tower_count_2km              float64
tower_count_5km              float64
tower_density_ratio          float64
digital_elevation_model      float64
region_enc                     int64
typeOfArea_enc                 int64
city_enc                       int64
demand_growth_pct            float64
year_trend                     int64
quarter                        int64
area_rolling_d_mbps          float64
tests                          int64
area_median_d                float64
area_median_u                float64
area_median_lat              float64
area_test_count                int64
distance_x_density           float64
dtype: object

In [39]:
import numpy as np

In [41]:
np.save('../outputs/models/X_train.npy', X_train)
np.save('../outputs/models/X_val.npy',   X_val)
np.save('../outputs/models/X_test.npy',  X_test)
np.save('../outputs/models/y_train.npy', y_train)
np.save('../outputs/models/y_val.npy',   y_val)
np.save('../outputs/models/y_test.npy',  y_test)

In [42]:
#Scalling features

In [44]:
from sklearn.preprocessing import StandardScaler

In [46]:
scaler_X = StandardScaler()
X_train_scal = scaler_X.fit_transform(X_train)
X_val_scal = scaler_X.transform(X_val)
X_test_scal = scaler_X.transform(X_test)

In [49]:
targets

['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']

In [53]:
scalers_y = {}

y_train_s = np.zeros_like(y_train, dtype=float)
y_val_s   = np.zeros_like(y_val, dtype=float)
y_test_s  = np.zeros_like(y_test, dtype=float)

for i, target in enumerate(targets):
    scaler = StandardScaler()
    
    y_train_s[:, i] = scaler.fit_transform(y_train.iloc[:, [i]]).ravel()
    y_val_s[:, i]   = scaler.transform(y_val.iloc[:, [i]]).ravel()
    y_test_s[:, i]  = scaler.transform(y_test.iloc[:, [i]]).ravel()
    
    scalers_y[target] = scaler

In [55]:
import pickle

with open('../outputs/models/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('../outputs/models/scalers_y.pkl', 'wb') as f:
    pickle.dump(scalers_y, f)

In [89]:
#Linear Regression Model

In [56]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [59]:
lr.fit(X_train_scal, y_train_s[:,2])

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [63]:
lr_val_arr = np.zeros((len(X_val), len(targets)))

y_pred_s = lr.predict(X_val_scal)
y_pred = scalers_y[target].inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
y_pred = np.clip(y_pred, 0, None)
lr_val_arr[:, 2] = y_pred

In [64]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [68]:
mae_lr = mean_absolute_error(y_val.iloc[:, 2], y_pred)
mae_lr

10.274093513845825

In [69]:
mape_lr = mean_absolute_percentage_error(y_val.iloc[:, 2], y_pred)
mape_lr

0.3579067888275682

In [70]:
lr_val_arr

array([[ 0.        ,  0.        , 21.53775458],
       [ 0.        ,  0.        , 27.04533866],
       [ 0.        ,  0.        , 18.13841652],
       ...,
       [ 0.        ,  0.        , 25.53260429],
       [ 0.        ,  0.        , 23.62667474],
       [ 0.        ,  0.        , 16.81374357]])

In [71]:
# Creating a loop to predict the multi targets (3)

In [84]:
lr_models = {}
val_preds = {}

In [85]:
lr_val_arr = np.zeros((len(X_val), len(targets)))
for i, target in enumerate(targets):
    lr = LinearRegression()
    lr.fit(X_train_scal, y_train_s[:,i])
    lr_models[target] = lr
    y_pred_s = lr.predict(X_val_scal)
    y_pred = scalers_y[target].inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
    y_pred = np.clip(y_pred, 0, None)
    lr_val_arr[:, i] = y_pred
    mae_lr = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_lr = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(mae_lr)
    print(mape_lr)

48.66452655778803
0.6263273445849337
6.296384101158351
0.5120303097034199
10.274093513845825
0.3579067888275682


In [79]:
lr_val_arr

array([[ 79.14228113,  15.33883973,  21.53775458],
       [208.15467647,  28.75664797,  27.04533866],
       [192.30291624,  24.51581936,  18.13841652],
       ...,
       [ 60.73803694,  12.69017345,  25.53260429],
       [225.00837396,  29.61163811,  23.62667474],
       [108.19883256,  14.26251602,  16.81374357]])

In [88]:
import joblib
val_preds['Linear Regression'] = lr_val_arr
joblib.dump(lr_models, '../outputs/models/lr_models.pkl')

['../outputs/models/lr_models.pkl']

In [90]:
#Decision Tree Model

In [91]:
from sklearn.tree import DecisionTreeRegressor

In [93]:
dt_models = {}
dt_val_arr = np.zeros((len(X_val), len(targets)))

In [111]:
for i, target in enumerate(targets):
    dt = DecisionTreeRegressor(max_depth=8, 
                               min_samples_leaf=15, 
                               random_state=42)
    dt.fit(X_train, y_train.iloc[:, i])
    dt_models[target] = dt
    y_pred = np.clip(dt.predict(X_val), 0, None)
    dt_val_arr[:, i] = y_pred
    mae_dt = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_dt = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_dt)
    print(mape_dt)

avg_d_mbps
49.29719761967411
0.6232099015858447
avg_u_mbps
6.835524345233845
0.5723874317382011
avg_lat_ms
10.788120877929655
0.37066555172287624


In [112]:
val_preds['Decision Tree'] = dt_val_arr
joblib.dump(dt_models, '../outputs/models/dt_models.pkl')

['../outputs/models/dt_models.pkl']

In [115]:
dt_val_arr

array([[ 86.16958915,  14.94332316,  20.90856863],
       [178.4631822 ,  32.86390786,  20.47611767],
       [222.94627729,  21.45123653,  20.47611767],
       ...,
       [ 52.95854472,  12.77337346,  26.13222836],
       [266.16371186,  38.65103   ,  26.13222836],
       [ 85.85975451,  12.77337346,  18.63985548]])

In [116]:
dt_val_arr.max()

np.float64(418.3155448717949)

In [117]:
lr_val_arr.max()

np.float64(349.01055650637335)

In [ ]:
#Random Forest Model

In [119]:
from sklearn.ensemble import RandomForestRegressor

In [120]:
rf_models  = {}
rf_val_arr = np.zeros((len(X_val), len(target)))

In [121]:
rf_val_arr

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [127]:
for i, target in enumerate(targets):
    rf = RandomForestRegressor(n_estimators = 500, 
                               max_depth= 15, 
                               min_samples_leaf= 4,
                               max_features = 'sqrt',
                               random_state = 42,
                               n_jobs = -1)
    rf.fit(X_train, y_train.iloc[:, i])
    rf_models[target] = rf
    y_pred = np.clip(rf.predict(X_val), 0, None)
    rf_val_arr[:, i] = y_pred
    mae_rf = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_rf = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_rf)
    print(mape_rf)

avg_d_mbps
46.37843958203325
0.5786352580631483
avg_u_mbps
6.3332115613170785
0.557732317551127
avg_lat_ms
10.022007085221919
0.34831399697247767


In [134]:
val_preds['Random Forest'] = rf_val_arr
joblib.dump(rf_models, '../outputs/models/rf_models.pkl')

['../outputs/models/rf_models.pkl']

In [135]:
rf_val_arr

array([[ 61.36247494,  12.32392648,  21.03744213, ...,   0.        ,
          0.        ,   0.        ],
       [190.4711968 ,  27.31970242,  20.63891682, ...,   0.        ,
          0.        ,   0.        ],
       [211.48774923,  25.70707839,  19.64242829, ...,   0.        ,
          0.        ,   0.        ],
       ...,
       [ 54.56434652,  15.09163867,  30.34503989, ...,   0.        ,
          0.        ,   0.        ],
       [247.66214406,  31.4762347 ,  29.00053765, ...,   0.        ,
          0.        ,   0.        ],
       [ 79.72499314,  15.8446307 ,  18.73990975, ...,   0.        ,
          0.        ,   0.        ]])

In [129]:
#Gradiant Boosting Model

In [130]:
from sklearn.ensemble import GradientBoostingRegressor

In [132]:
gb_models  = {}
gb_val_arr = np.zeros((len(X_val), len(targets)))

In [142]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators = 200,
        learning_rate = 0.05,
        max_depth = 4,
        random_state = 42,
    )
    gb.fit(X_train, y_train.iloc[:, i])
    gb_models[target] = gb
    y_pred = np.clip(gb.predict(X_val), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
45.74118100880731
0.5177998771368032
avg_u_mbps
6.206352327287999
0.5203321722515499
avg_lat_ms
9.775094804651202
0.34519778967762804


In [146]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators =200,
        learning_rate = 0.03,
        max_depth = 5,
        subsample= 0.8,
        min_samples_leaf=4,
        random_state = 42,
    )
    gb.fit(X_train, y_train.iloc[:, i])
    gb_models[target] = gb
    y_pred = np.clip(gb.predict(X_val), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
45.63811220595615
0.514674425062059
avg_u_mbps
6.2126044215515055
0.5114128758651089
avg_lat_ms
9.874245256333515
0.3407951055436228


In [159]:
#model enhancement

In [161]:
#using log1p to log transform the y in order to fix the extreme difference in the speeds (0 - 2000), \
#where rare extreme values would dominate the model's learning.

In [164]:
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

In [166]:
y_train_log.head(2)

,avg_d_mbps,avg_u_mbps,avg_lat_ms
4211,5.469222,3.563430,2.995732
5786,5.593569,3.289476,2.867899


In [167]:
y_train.head(2)

,avg_d_mbps,avg_u_mbps,avg_lat_ms
4211,236.2755,34.2840,19.0
5786,267.6930,25.8288,16.6


In [169]:
y_train_log.max()

avg_d_mbps    7.194235
avg_u_mbps    4.758775
avg_lat_ms    6.350886
dtype: float64

In [170]:
y_train.max()

avg_d_mbps    1330.731
avg_u_mbps     115.603
avg_lat_ms     572.000
dtype: float64

In [250]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators =200,
        learning_rate = 0.05,
        max_depth = 4,
        subsample= 0.8,
        random_state = 42,
    )
    gb.fit(X_train, y_train_log.iloc[:, i])
    gb_models[target] = gb
    y_pred_log = gb.predict(X_val)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
44.20465938571749
0.4170184684245479
avg_u_mbps
6.315505992926719
0.44374651685355043
avg_lat_ms
8.943491679069304
0.26988516994273287


In [202]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators =200,
        learning_rate = 0.03,
        max_depth = 5,
        subsample= 0.8,
        min_samples_leaf=4,
        random_state = 42,
    )
    gb.fit(X_train, y_train_log.iloc[:, i])
    gb_models[target] = gb
    y_pred_log = gb.predict(X_val)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
44.37463919576045
0.4171568098098987
avg_u_mbps
6.238923393646033
0.45217998173602253
avg_lat_ms
8.940820338920275
0.2668596197246675


In [203]:
gb_models

{'avg_d_mbps': GradientBoostingRegressor(learning_rate=0.03, max_depth=5, min_samples_leaf=4,
                           n_estimators=200, random_state=42, subsample=0.8),
 'avg_u_mbps': GradientBoostingRegressor(learning_rate=0.03, max_depth=5, min_samples_leaf=4,
                           n_estimators=200, random_state=42, subsample=0.8),
 'avg_lat_ms': GradientBoostingRegressor(learning_rate=0.03, max_depth=5, min_samples_leaf=4,
                           n_estimators=200, random_state=42, subsample=0.8)}

In [204]:
val_preds

{'Linear Regression': array([[ 79.14228113,  15.33883973,  21.53775458],
        [208.15467647,  28.75664797,  27.04533866],
        [192.30291624,  24.51581936,  18.13841652],
        ...,
        [ 60.73803694,  12.69017345,  25.53260429],
        [225.00837396,  29.61163811,  23.62667474],
        [108.19883256,  14.26251602,  16.81374357]]),
 'Decision Tree': array([[ 86.16958915,  14.94332316,  20.90856863],
        [178.4631822 ,  32.86390786,  20.47611767],
        [222.94627729,  21.45123653,  20.47611767],
        ...,
        [ 52.95854472,  12.77337346,  26.13222836],
        [266.16371186,  38.65103   ,  26.13222836],
        [ 85.85975451,  12.77337346,  18.63985548]]),
 'Random Forest': array([[ 61.36247494,  12.32392648,  21.03744213, ...,   0.        ,
           0.        ,   0.        ],
        [190.4711968 ,  27.31970242,  20.63891682, ...,   0.        ,
           0.        ,   0.        ],
        [211.48774923,  25.70707839,  19.64242829, ...,   0.        ,
     

In [252]:
val_preds['Gradient Boosting'] = gb_val_arr
joblib.dump(gb_models, '../outputs/models/gb_models.pkl')

['../outputs/models/gb_models.pkl']

In [192]:
#Tune gb

In [194]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':     [3, 4, 5],
    'subsample':     [0.7, 0.8, 1.0],
}

gb_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,                          # 5-fold cross-validation on training set
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1,
)
gb_search.fit(X_train, y_train_log.iloc[:, 0]) 

Fitting 5 folds for each of 81 candidates, totalling 405 fits


,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'n_estimators': [100, 200, ...], 'subsample': [0.7, 0.8, ...]}"
,scoring,'neg_mean_absolute_percentage_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [198]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators =100,
        learning_rate = 0.05,
        max_depth = 5,
        min_samples_split= 2,
        random_state = 42,
    )
    gb.fit(X_train, y_train_log.iloc[:, i])
    gb_models[target] = gb
    y_pred_log = gb.predict(X_val)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
44.01863257331735
0.4330386458041031
avg_u_mbps
6.259390893803309
0.4548243902454858
avg_lat_ms
8.738657911330765
0.2605910010592049


In [201]:
for i, target in enumerate(targets):
    gb = GradientBoostingRegressor(
        n_estimators =200,
        learning_rate = 0.05,
        max_depth = 5,
        min_samples_split= 2,
        random_state = 42,
    )
    gb.fit(X_train, y_train_log.iloc[:, i])
    gb_models[target] = gb
    y_pred_log = gb.predict(X_val)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    gb_val_arr[:, i] = y_pred
    mae_gb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_gb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_gb)
    print(mape_gb)

avg_d_mbps
44.54364049488046
0.44201027030581713
avg_u_mbps
6.277877049281234
0.4550385380842369
avg_lat_ms
8.904311424021952
0.26847495708907837


In [240]:
#XGBoost model

In [241]:
from xgboost import XGBRegressor

In [242]:
xgb_models  = {}
xgb_val_arr = np.zeros((len(X_val), len(targets)))

In [244]:
for i, target in enumerate(targets):
    xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    xgb.fit(X_train, y_train.iloc[:, i])
    xgb_models[target] = xgb
    y_pred = np.clip(xgb.predict(X_val), 0, None)
    xgb_val_arr[:, i] = y_pred
    mae_xgb = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_xgb = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_xgb)
    print(mape_xgb)

avg_d_mbps
46.18868144891847
0.5123706661602241
avg_u_mbps
6.343584116229342
0.5135035497927651
avg_lat_ms
9.782672042982302
0.33611162642269904


In [245]:
from sklearn.ensemble import ExtraTreesRegressor

In [246]:
ext_models  = {}
ext_val_arr = np.zeros((len(X_val), len(targets)))

In [249]:
for i, target in enumerate(targets):
    ext = ExtraTreesRegressor(
        n_estimators=200,
        random_state=42
    )
    ext.fit(X_train, y_train.iloc[:, i])
    ext_models[target] = ext
    y_pred = np.clip(ext.predict(X_val), 0, None)
    ext_val_arr[:, i] = y_pred
    mae_ext = mean_absolute_error(y_val.iloc[:, i], y_pred)
    mape_ext = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred)
    print(target)
    print(mae_ext)
    print(mape_ext)

avg_d_mbps
46.98312566312386
0.550231047030455
avg_u_mbps
6.586024158188431
0.6060858338240896
avg_lat_ms
10.474070291203839
0.37489114405557444


In [195]:
#Selecting the winner model

In [196]:
model_names = ['Linear Regression', 'Decision Tree','Random Forest', 'Gradient Boosting']

In [206]:
from sklearn.metrics import r2_score, mean_squared_error

In [209]:
all_val_metrics = {}
for name in model_names:
    all_val_metrics[name] = {}
    for i, target in enumerate(targets):
        yt = y_val.iloc[:, i]
        yp = val_preds[name][:, i]
        valid = yt > 0
        all_val_metrics[name][target] = {
            'MAPE': round(float(mean_absolute_percentage_error(yt[valid], yp[valid])) * 100, 1),
            'R2':   round(float(r2_score(yt, yp)), 4),
            'RMSE': round(float(np.sqrt(mean_squared_error(yt, yp))), 3),
            'MAE':  round(float(mean_absolute_error(yt, yp)), 3),
        }

In [211]:
metric_weights = [
    ('MAPE', 3, True),
    ('R2',   2, False),
    ('RMSE', 1, True),
    ('MAE',  1, True),
]

In [212]:
metric_weights

[('MAPE', 3, True), ('R2', 2, False), ('RMSE', 1, True), ('MAE', 1, True)]

In [218]:
total_rank = {name: 0 for name in model_names}
for target in targets:
    for metric, weight, lower_better in metric_weights:
        ordered = sorted(model_names,
                         key=lambda n: all_val_metrics[n][target][metric],
                         reverse=not lower_better)
        for pos, name in enumerate(ordered, start=1):
            total_rank[name] += pos * weight
 
ranked      = sorted(total_rank.items(), key=lambda x: x[1])
winner_name = ranked[0][0]

In [215]:
ranked

[('Gradient Boosting', 36),
 ('Random Forest', 37),
 ('Linear Regression', 59),
 ('Decision Tree', 78)]

In [216]:
total_rank

{'Linear Regression': 59,
 'Decision Tree': 78,
 'Random Forest': 37,
 'Gradient Boosting': 36}

In [217]:
winner_name

'Gradient Boosting'

In [228]:
models_stg = ['', '', '', '']

In [230]:
for idx, (name, score) in enumerate(ranked):
    d_mape = all_val_metrics[name]['avg_d_mbps']['MAPE']
    u_mape = all_val_metrics[name]['avg_u_mbps']['MAPE']
    arrow  = 'WINNER' if idx == 0 else ''

In [231]:
winner_name

'Gradient Boosting'

In [232]:
with open('../outputs/models/best_model_name.txt', 'w') as f:
    f.write(winner_name)

In [235]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('../scripts/architecture.py')))
from architecture import NeuralNetworkRegressor